<a href="https://colab.research.google.com/github/Jimena1011/Asistencia/blob/main/Vehicle_Detection_with_Yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jimenavargasvega_video_of_vehicle_path = kagglehub.dataset_download('jimenavargasvega/video-of-vehicle')
jimenavargasvega_modelo_para_entrenar_el_programa_path = kagglehub.dataset_download('jimenavargasvega/modelo-para-entrenar-el-programa')

print('Data source import complete.')


100%|██████████| 4.46G/4.46G [00:54<00:00, 88.5MB/s]

Extracting files...


100%|██████████| 215M/215M [00:01<00:00, 131MB/s]

Extracting files...


Data source import complete.


In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CELDA 1 — Verificar GPU
# ═══════════════════════════════════════════════════════════════════════════
!nvidia-smi


Sat Jun  6 07:04:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# CELDA 2 — Instalar / actualizar ultralytics (YOLO26 requiere versión reciente)
# ═══════════════════════════════════════════════════════════════════════════
!pip install ultralytics -q
!pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install sahi -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 27.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 113.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 145.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 47.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 21.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# ============================================================
# CELDA 1 — Detección + Tracking BoT-SORT + Conteo con línea
# Versión: Colab  |  Cambiar paths si se migra a Kaggle
# ============================================================
import os, csv, time, cv2
import numpy as np
import torch
import pandas as pd
import kagglehub
from collections import deque
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

CLASS_NAMES  = {0:"persona", 1:"bicicleta", 2:"carro", 3:"motocicleta", 4:"bus", 5:"truck"}
CLASS_COLORS = {0:(255,128,0), 1:(0,255,255), 2:(0,255,0), 3:(255,0,255), 4:(255,0,0), 5:(0,128,255)}

CONF_PER_CLASS = {
    0: 0.40,   # persona
    1: 0.25,   # bicicleta
    2: 0.30,   # carro
    3: 0.15,   # motocicleta
    4: 0.35,   # bus
    5: 0.70,   # truck
}

# ── Paths (Colab) ─────────────────────────────────────────
model_dir  = kagglehub.dataset_download('jimenavargasvega/modelo-para-entrenar-el-programa')
MODEL_PATH = os.path.join(model_dir, 'best_visdrone_yolo26m_v1.pt')

video_dir  = kagglehub.dataset_download('jimenavargasvega/video-of-vehicle')
VIDEOS     = [os.path.join(video_dir, 'DJI_0251.MOV')]
# ──────────────────────────────────────────────────────────

ROI_POLYGON = np.array([
    [ 800,  330],
    [1400,  330],
    [1500, 1050],
    [ 600, 1050],
], dtype=np.int32)

# Zona de conteo: dos líneas paralelas Y=580 y Y=620, X de 600 a 1500
# La banda de 40px cubre gaps de detección cortos sin generar falsos cruces
COUNTING_LINE_A = [(600, 580), (1500, 580)]   # línea superior
COUNTING_LINE_B = [(600, 620), (1500, 620)]   # línea inferior
ZONE_Y_TOP    = 580
ZONE_Y_BOTTOM = 620

# Estela: máximo de puntos guardados por ID (mínimo para no saturar CPU)
TRAIL_MAXLEN = 8


# ── Helpers de geometría (del código viejo) ───────────────
def ccw(A, B, C):
    return (C[1]-A[1]) * (B[0]-A[0]) > (B[1]-A[1]) * (C[0]-A[0])

def intersect(A, B, C, D):
    return ccw(A,C,D) != ccw(B,C,D) and ccw(A,B,C) != ccw(A,B,D)

def get_direction(p1, p2):
    """Devuelve 'Sur' si el objeto baja (Y aumenta), 'Norte' si sube."""
    if p1[1] < p2[1]:
        return "Sur"
    elif p1[1] > p2[1]:
        return "Norte"
    return "Indefinido"
# ──────────────────────────────────────────────────────────


def process_video(video_path, model_path):
    if not os.path.exists(video_path):
        print(f"No encontrado: {video_path}")
        return

    model = YOLO(model_path)
    model.to(DEVICE)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    out_path   = f'/content/{video_name}_tracking_doble_linea.mp4'
    csv_path   = f'/content/{video_name}_conteo.csv'

    cap    = cv2.VideoCapture(video_path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\n{'='*60}")
    print(f"Video : {video_name}")
    print(f"  {width}x{height} @ {fps:.1f}fps | {total} frames | {total/fps/60:.1f} min")
    print(f"  Tracker : BoT-SORT  |  Zona Y=580–620  |  Trail={TRAIL_MAXLEN}pts")
    print(f"{'='*60}")

    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    csv_file   = open(csv_path, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['frame', 'track_id', 'clase', 'clase_id',
                         'confianza', 'cx', 'cy', 'direccion', 'cruce'])

    # ── Estado del tracker ────────────────────────────────
    trail        = {}          # trail[track_id] = deque de centroides
    counted_ids  = set()       # IDs que ya cruzaron la línea (no duplicar)
    counter_sur  = {}          # {clase: n} dirección Sur  (bajan / salen)
    counter_norte = {}         # {clase: n} dirección Norte (suben / entran)
    # ──────────────────────────────────────────────────────

    start_time = time.time()
    frame_idx  = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # ── Tracking con BoT-SORT ──────────────────────────
        results = model.track(
            frame,
            conf      = min(CONF_PER_CLASS.values()),
            imgsz     = 640,
            verbose   = False,
            device    = DEVICE,
            half      = True,
            tracker   = "botsort.yaml",
            persist   = True,       # mantiene IDs entre frames
        )
        # ──────────────────────────────────────────────────

        # Dibujar ROI
        cv2.polylines(frame, [ROI_POLYGON], isClosed=True, color=(255, 0, 0), thickness=2)

        # Zona de conteo: dos líneas paralelas
        cv2.line(frame, COUNTING_LINE_A[0], COUNTING_LINE_A[1], (46, 162, 112), 1)
        cv2.line(frame, COUNTING_LINE_B[0], COUNTING_LINE_B[1], (46, 162, 112), 1)

        # ── Limpiar trails de IDs que ya no están activos ──
        active_ids = set()

        for r in results:
            if r.boxes is None or len(r.boxes) == 0:
                continue
            if r.boxes.id is None:
                continue

            for box, track_id_t, cls_t, conf_t in zip(
                r.boxes.xyxy,
                r.boxes.id,
                r.boxes.cls,
                r.boxes.conf
            ):
                cls    = int(cls_t)
                conf   = float(conf_t)
                tid    = int(track_id_t)

                # Filtro confianza por clase
                if conf < CONF_PER_CLASS.get(cls, 0.3):
                    continue

                x1, y1, x2, y2 = map(int, box.tolist())
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Filtro ROI
                if cv2.pointPolygonTest(ROI_POLYGON, (cx, cy), False) < 0:
                    continue

                active_ids.add(tid)
                cls_name = CLASS_NAMES.get(cls, str(cls))
                color    = CLASS_COLORS.get(cls, (200, 200, 200))

                # ── Trail (estela mínima) ──────────────────
                if tid not in trail:
                    trail[tid] = deque(maxlen=TRAIL_MAXLEN)
                trail[tid].appendleft((cx, cy))

                # ── Lógica de cruce de zona doble ─────────────
                # Se cuenta cuando el ID cruza cualquiera de las dos líneas,
                # o cuando su centroide aparece dentro de la banda (cubre gaps)
                cruce = ""
                if tid not in counted_ids:
                    en_zona = ZONE_Y_TOP <= cy <= ZONE_Y_BOTTOM

                    cruzó_linea = False
                    direction   = ""
                    if len(trail[tid]) >= 2:
                        p_prev = trail[tid][1]
                        p_curr = trail[tid][0]
                        if (intersect(p_prev, p_curr, COUNTING_LINE_A[0], COUNTING_LINE_A[1]) or
                                intersect(p_prev, p_curr, COUNTING_LINE_B[0], COUNTING_LINE_B[1])):
                            cruzó_linea = True
                            direction   = get_direction(p_prev, p_curr)

                        # Caso gap: reapareció del otro lado sin cruzar en dos frames consecutivos
                        elif en_zona:
                            # Si el punto anterior estaba claramente fuera de la zona,
                            # inferimos la dirección por de qué lado venía
                            if p_prev[1] < ZONE_Y_TOP:
                                cruzó_linea = True
                                direction   = "Sur"
                            elif p_prev[1] > ZONE_Y_BOTTOM:
                                cruzó_linea = True
                                direction   = "Norte"

                    if cruzó_linea and direction:
                        counted_ids.add(tid)
                        cruce = direction
                        # Iluminar zona al detectar cruce
                        cv2.line(frame, COUNTING_LINE_A[0], COUNTING_LINE_A[1], (255, 255, 255), 2)
                        cv2.line(frame, COUNTING_LINE_B[0], COUNTING_LINE_B[1], (255, 255, 255), 2)
                        if direction == "Sur":
                            counter_sur[cls_name]    = counter_sur.get(cls_name, 0) + 1
                        elif direction == "Norte":
                            counter_norte[cls_name]  = counter_norte.get(cls_name, 0) + 1
                        print(f"  [CRUCE] frame={frame_idx} ID={tid} clase={cls_name} dir={direction}")

                # ── Dibujar bounding box ───────────────────
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 1)
                label  = f"ID{tid} {cls_name} {conf:.2f}"
                t_size = cv2.getTextSize(label, 0, fontScale=0.25, thickness=1)[0]
                cv2.rectangle(frame, (x1, y1), (x1+t_size[0], y1-t_size[1]-3), color, -1)
                cv2.putText(frame, label, (x1, y1-2), 0, 0.25, [225,255,255], 1, cv2.LINE_4)

                # ── Dibujar trail (cada 2do punto para ahorrar CPU) ──
                pts = list(trail[tid])
                for i in range(1, len(pts), 2):
                    if pts[i-1] is None or pts[i] is None:
                        continue
                    cv2.line(frame, pts[i-1], pts[i], color, 1)

                # ── CSV por frame ──────────────────────────
                csv_writer.writerow([frame_idx, tid, cls_name, cls,
                                     f"{conf:.4f}", cx, cy, "", cruce])

        # Limpiar trails de IDs que desaparecieron
        for old_id in list(trail.keys()):
            if old_id not in active_ids:
                del trail[old_id]

        # ── Paneles de conteo en pantalla ──────────────────
        # Panel Sur (izquierda)
        cv2.rectangle(frame, (10, 15), (250, 45), (85, 45, 255), -1, cv2.LINE_4)
        cv2.putText(frame, 'Salida (Sur)', (15, 35), 0, 0.65, [225,255,255], 1, cv2.LINE_4)
        for idx, (cls_n, val) in enumerate(counter_sur.items()):
            cv2.rectangle(frame, (10, 50+(idx*32)), (200, 80+(idx*32)), (85,45,255), -1, cv2.LINE_4)
            cv2.putText(frame, f"{cls_n}: {val}", (15, 72+(idx*32)), 0, 0.7, [255,255,255], 1, cv2.LINE_4)

        # Panel Norte (derecha)
        cv2.rectangle(frame, (width-260, 15), (width-10, 45), (180, 90, 0), -1, cv2.LINE_4)
        cv2.putText(frame, 'Entrada (Norte)', (width-255, 35), 0, 0.65, [225,255,255], 1, cv2.LINE_4)
        for idx, (cls_n, val) in enumerate(counter_norte.items()):
            cv2.rectangle(frame, (width-260, 50+(idx*32)), (width-10, 80+(idx*32)), (180,90,0), -1, cv2.LINE_4)
            cv2.putText(frame, f"{cls_n}: {val}", (width-255, 72+(idx*32)), 0, 0.7, [255,255,255], 1, cv2.LINE_4)
        # ──────────────────────────────────────────────────

        out.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            elapsed  = time.time() - start_time
            fps_real = frame_idx / max(elapsed, 1e-6)
            eta      = (total - frame_idx) / fps_real if fps_real > 0 else 0
            print(f"\rFrame {frame_idx:>6}/{total} ({frame_idx/total*100:5.1f}%)  |  "
                  f"Tiempo: {time.strftime('%H:%M:%S', time.gmtime(elapsed))}  |  "
                  f"ETA: {time.strftime('%H:%M:%S', time.gmtime(eta))}  |  "
                  f"{fps_real:.1f} fps", end="", flush=True)

    cap.release()
    out.release()
    csv_file.close()

    # ── Convertir a H.264 ─────────────────────────────────
    elapsed = time.time() - start_time
    print(f"\n\nCompletado en {time.strftime('%H:%M:%S', time.gmtime(elapsed))}")
    print(f"  FPS reales: {total/elapsed:.2f}")

    out_h264 = out_path.replace('.mp4', '_h264.mp4')
    os.system(f'ffmpeg -y -i "{out_path}" -vcodec libx264 -crf 23 -preset fast "{out_h264}" -loglevel quiet')
    if os.path.exists(out_h264):
        os.replace(out_h264, out_path)
        print(f"  Video H.264: {out_path}")

    # ── Resumen final ─────────────────────────────────────
    total_sur   = sum(counter_sur.values())
    total_norte = sum(counter_norte.values())

    print(f"\n{'='*60}")
    print(f"RESUMEN DE CONTEO — {video_name}")
    print(f"{'='*60}")
    print(f"  Salida (Sur)   — {total_sur} vehículos")
    for cls_n, val in counter_sur.items():
        print(f"    {cls_n:>12}: {val}")
    print(f"  Entrada (Norte) — {total_norte} vehículos")
    for cls_n, val in counter_norte.items():
        print(f"    {cls_n:>12}: {val}")
    print(f"  IDs únicos contados: {len(counted_ids)}")
    print(f"  CSV guardado: {csv_path}")


for vp in VIDEOS:
    process_video(vp, MODEL_PATH)

print(f"\n{'='*60}")
print("TODOS LOS VIDEOS PROCESADOS")
print(f"{'='*60}")


Device: cuda
Using Colab cache for faster access to the 'modelo-para-entrenar-el-programa' dataset.
Using Colab cache for faster access to the 'video-of-vehicle' dataset.

Video : DJI_0251
  1920x1080 @ 30.0fps | 5056 frames | 2.8 min
  Tracker : BoT-SORT  |  Zona Y=580–620  |  Trail=8pts
Frame     30/5056 (  0.6%)  |  Tiempo: 00:00:02  |  ETA: 00:06:44  |  12.4 fps  [CRUCE] frame=50 ID=5 clase=carro dir=Sur
Frame    120/5056 (  2.4%)  |  Tiempo: 00:00:09  |  ETA: 00:06:15  |  13.2 fps  [CRUCE] frame=120 ID=3 clase=carro dir=Sur
Frame    240/5056 (  4.7%)  |  Tiempo: 00:00:18  |  ETA: 00:06:02  |  13.3 fps  [CRUCE] frame=256 ID=15 clase=carro dir=Sur
Frame    330/5056 (  6.5%)  |  Tiempo: 00:00:24  |  ETA: 00:05:57  |  13.2 fps  [CRUCE] frame=332 ID=16 clase=carro dir=Sur
Frame    390/5056 (  7.7%)  |  Tiempo: 00:00:29  |  ETA: 00:05:52  |  13.2 fps  [CRUCE] frame=391 ID=1337 clase=persona dir=Sur
Frame    450/5056 (  8.9%)  |  Tiempo: 00:00:34  |  ETA: 00:05:48  |  13.2 fps  [CRUCE] f